# 02 - Pandas con datos reales: NYC Yellow Taxi 2015

**Pregunta de negocio:** *¿Cual es el perfil tipico de un viaje en taxi en NYC?*

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

## Objetivos
- Cargar una muestra de ~100K viajes desde BigQuery (con cache)
- Limpieza de datos: zonas TLC invalidas, tarifas negativas, duraciones imposibles
- Crear columnas derivadas: duracion, velocidad, porcentaje de propina
- Perfilado de datos: describe(), dtypes, memory_usage()
- Indexacion temporal con DatetimeIndex y resample

In [1]:
import sys
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)

sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper

bq = BigQueryHelper()
print(f"Pandas version: {pd.__version__}")

✓ Conectado a BigQuery - Proyecto: gen-lang-client-0180273702
✓ Cache en: /home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos/data/nyc_taxi/cache
Pandas version: 2.3.3


## 1. Cargar muestra de ~100K viajes desde BigQuery

Usamos `FARM_FINGERPRINT` para un muestreo reproducible y descargamos solo las columnas que necesitamos. El helper guarda el resultado en cache (parquet) para no repetir la consulta.

**Nota:** Esta tabla usa `pickup_location_id` y `dropoff_location_id` (IDs de zonas TLC, enteros 1-263) en lugar de coordenadas GPS.

In [2]:
query_sample = """
SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    pickup_location_id,
    dropoff_location_id,
    rate_code,
    payment_type,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
ORDER BY FARM_FINGERPRINT(
    CAST(pickup_datetime AS STRING) || CAST(dropoff_datetime AS STRING)
)
LIMIT 100000
"""

df_raw = bq.run_query(query_sample, label="sample_100k_yellow_2015")
print(f"\nShape: {df_raw.shape}")
df_raw.head()

📊 Estimación: 16.671 GB → $0.1018 USD
⏳ Ejecutando query...


/home/danielmf31/Documentos/Documentos_Trabajo/Ingenieria/Programacion/VSCode/Proyectos_personales/Ciencia_Datos/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✓ Completado: 100,000 filas en 21.7s
💾 Cache guardado: sample_100k_yellow_2015_942b6a908e1c.parquet (2.6 MB)

Shape: (100000, 13)


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,pickup_location_id,dropoff_location_id,rate_code,payment_type,fare_amount,tip_amount,tolls_amount,total_amount
0,2,2015-10-12 20:38:06+00:00,2015-10-12 20:45:52+00:00,1,1.22,48,186,1,1,7.00,2.08,0.00,10.38
1,2,2015-10-12 20:56:55+00:00,2015-10-12 21:02:03+00:00,5,0.57,125,125,1,2,5.00,0.00,0.00,6.30
2,2,2015-10-25 17:46:46+00:00,2015-10-25 17:52:20+00:00,1,0.89,100,230,1,2,6.00,0.00,0.00,6.80
3,1,2015-02-23 17:20:36+00:00,2015-02-23 17:32:58+00:00,2,1.40,237,262,1,2,9.00,0.00,0.00,10.80
4,2,2015-01-25 12:30:26+00:00,2015-01-25 12:48:12+00:00,2,3.52,137,143,1,1,15.00,3.16,0.00,18.96


## 2. Perfilado inicial (antes de limpiar)

Antes de modificar nada, exploramos los datos crudos para entender que tenemos: tipos de datos, valores nulos, estadisticas basicas y uso de memoria.

In [3]:
# Tipos de datos y uso de memoria
print("=== Tipos de datos ===")
print(df_raw.dtypes)
print(f"\n=== Uso de memoria ===")
mem = df_raw.memory_usage(deep=True)
print(mem)
print(f"\nTotal: {mem.sum() / 1024**2:.2f} MB")

=== Tipos de datos ===
vendor_id                           object
pickup_datetime        datetime64[us, UTC]
dropoff_datetime       datetime64[us, UTC]
passenger_count                      Int64
trip_distance                      float64
pickup_location_id                  object
dropoff_location_id                 object
rate_code                           object
payment_type                        object
fare_amount                        float64
tip_amount                         float64
tolls_amount                       float64
total_amount                       float64
dtype: object

=== Uso de memoria ===
Index                      132
vendor_id              5000000
pickup_datetime         800000
dropoff_datetime        800000
passenger_count         900000
trip_distance           800000
pickup_location_id     5180259
dropoff_location_id    5176725
rate_code              5000005
payment_type           5000000
fare_amount             800000
tip_amount              800000
tolls_am

In [4]:
# Estadisticas descriptivas
df_raw.describe()

,passenger_count,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,1.68,83.03,12.91,1.71,0.31,16.03
std,1.34,25314.10,10.80,2.52,1.49,13.28
min,0.00,0.00,-52.00,0.00,0.00,-52.80
25%,1.00,1.00,6.50,0.00,0.00,8.75
50%,1.00,1.72,9.50,1.16,0.00,11.80
75%,2.00,3.20,14.50,2.32,0.00,17.80
max,6.00,8005023.20,255.55,156.00,150.08,307.62


In [5]:
# Valores nulos por columna
nulos = df_raw.isnull().sum()
nulos_pct = (nulos / len(df_raw) * 100).round(2)
pd.DataFrame({"nulos": nulos, "porcentaje": nulos_pct}).query("nulos > 0")

,nulos,porcentaje


In [6]:
# Distribucion de columnas categoricas
print("=== payment_type ===")
print(df_raw["payment_type"].value_counts())
print(f"\n=== rate_code ===")
print(df_raw["rate_code"].value_counts())
print(f"\n=== passenger_count ===")
print(df_raw["passenger_count"].value_counts().sort_index())
print(f"\n=== pickup_location_id (top 10) ===")
print(df_raw["pickup_location_id"].value_counts().head(10))

=== payment_type ===
payment_type
1    62433
2    37129
3      329
4      109
Name: count, dtype: int64

=== rate_code ===
rate_code
1     97390
2      2054
5       333
3       187
4        30
99        5
6         1
Name: count, dtype: int64

=== passenger_count ===
passenger_count
0       36
1    70404
2    14233
3     4258
4     2053
5     5518
6     3498
Name: count, dtype: Int64

=== pickup_location_id (top 10) ===
pickup_location_id
237    3685
161    3496
79     3409
234    3266
230    3264
162    3248
236    3238
170    3233
186    3202
48     3135
Name: count, dtype: int64


## 3. Limpieza de datos

Aplicamos filtros para eliminar registros invalidos. Cada paso se documenta con el numero de filas eliminadas y la razon.

### 3.1 Zonas TLC invalidas

Los `pickup_location_id` y `dropoff_location_id` son IDs de zonas TLC (Taxi and Limousine Commission) con valores validos entre 1 y 263. Registros con valores nulos o fuera de este rango indican datos corruptos.

In [7]:
# Empezamos con una copia para no modificar los datos crudos
df = df_raw.copy()
n_inicial = len(df)
print(f"Filas iniciales: {n_inicial:,}")

# --- 3.1 Zonas TLC invalidas ---
# Convertir a numerico (pueden venir como STRING desde BigQuery)
df["pickup_location_id"] = pd.to_numeric(df["pickup_location_id"], errors="coerce")
df["dropoff_location_id"] = pd.to_numeric(df["dropoff_location_id"], errors="coerce")

zonas_validas = (
    df["pickup_location_id"].notna() &
    df["dropoff_location_id"].notna() &
    df["pickup_location_id"].between(1, 263) &
    df["dropoff_location_id"].between(1, 263)
)

n_zonas_invalidas = (~zonas_validas).sum()
print(f"\nZonas TLC invalidas: {n_zonas_invalidas:,} filas ({n_zonas_invalidas/n_inicial*100:.1f}%)")

df = df[zonas_validas]
print(f"Filas restantes: {len(df):,}")

Filas iniciales: 100,000

Zonas TLC invalidas: 1,977 filas (2.0%)
Filas restantes: 98,023


### 3.2 Tarifas negativas, distancias cero y duraciones imposibles

Un viaje valido debe tener:
- `fare_amount > 0`
- `trip_distance > 0`
- `total_amount > 0`
- Duracion entre 1 minuto y 6 horas (360 min)

In [8]:
n_antes = len(df)

# Calcular duracion para usarla en el filtro
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])
df["duration_min"] = (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds() / 60

# Filtros de negocio
fare_ok = df["fare_amount"] > 0
dist_ok = df["trip_distance"] > 0
total_ok = df["total_amount"] > 0
dur_ok = df["duration_min"].between(1, 360)

# Diagnostico antes de filtrar
print("Filas eliminadas por cada filtro (pueden solaparse):")
print(f"  fare_amount <= 0:    {(~fare_ok).sum():,}")
print(f"  trip_distance <= 0:  {(~dist_ok).sum():,}")
print(f"  total_amount <= 0:   {(~total_ok).sum():,}")
print(f"  duracion fuera [1, 360] min: {(~dur_ok).sum():,}")

df = df[fare_ok & dist_ok & total_ok & dur_ok]
n_eliminadas = n_antes - len(df)
print(f"\nTotal eliminadas: {n_eliminadas:,} ({n_eliminadas/n_antes*100:.1f}%)")
print(f"Filas restantes: {len(df):,}")

Filas eliminadas por cada filtro (pueden solaparse):
  fare_amount <= 0:    40
  trip_distance <= 0:  346
  total_amount <= 0:   30
  duracion fuera [1, 360] min: 613

Total eliminadas: 731 (0.7%)
Filas restantes: 97,292


### 3.3 Valores nulos restantes

Estrategia: eliminamos filas con nulos en columnas criticas. Para `passenger_count` nulo, imputamos con la moda (1 pasajero es lo mas comun).

In [9]:
n_antes = len(df)

# Nulos en columnas criticas -> eliminar
cols_criticas = ["pickup_datetime", "dropoff_datetime", "fare_amount", "trip_distance"]
df = df.dropna(subset=cols_criticas)

# passenger_count: imputar con moda
if df["passenger_count"].isnull().any():
    moda_pasajeros = df["passenger_count"].mode()[0]
    n_imputados = df["passenger_count"].isnull().sum()
    df["passenger_count"] = df["passenger_count"].fillna(moda_pasajeros)
    print(f"passenger_count: {n_imputados:,} nulos imputados con moda = {moda_pasajeros}")

n_eliminadas = n_antes - len(df)
print(f"Filas eliminadas por nulos criticos: {n_eliminadas:,}")
print(f"Filas restantes: {len(df):,}")

# Verificar que no quedan nulos
print(f"\nNulos restantes:\n{df.isnull().sum()}")

Filas eliminadas por nulos criticos: 0
Filas restantes: 97,292

Nulos restantes:
vendor_id              0
pickup_datetime        0
dropoff_datetime       0
passenger_count        0
trip_distance          0
pickup_location_id     0
dropoff_location_id    0
rate_code              0
payment_type           0
fare_amount            0
tip_amount             0
tolls_amount           0
total_amount           0
duration_min           0
dtype: int64


### 3.4 Resumen de limpieza: antes vs despues

In [10]:
n_final = len(df)
n_eliminadas_total = n_inicial - n_final
pct_eliminadas = n_eliminadas_total / n_inicial * 100

print("=" * 50)
print("RESUMEN DE LIMPIEZA")
print("=" * 50)
print(f"Filas originales:   {n_inicial:>10,}")
print(f"Filas eliminadas:   {n_eliminadas_total:>10,} ({pct_eliminadas:.1f}%)")
print(f"Filas limpias:      {n_final:>10,} ({100 - pct_eliminadas:.1f}%)")
print("=" * 50)

# Comparar describe antes y despues
print("\n--- Estadisticas ANTES de limpiar ---")
print(df_raw[["fare_amount", "trip_distance", "tip_amount"]].describe().to_string())
print("\n--- Estadisticas DESPUES de limpiar ---")
print(df[["fare_amount", "trip_distance", "tip_amount"]].describe().to_string())

RESUMEN DE LIMPIEZA
Filas originales:      100,000
Filas eliminadas:        2,708 (2.7%)
Filas limpias:          97,292 (97.3%)

--- Estadisticas ANTES de limpiar ---
       fare_amount  trip_distance  tip_amount
count    100000.00      100000.00   100000.00
mean         12.91          83.03        1.71
std          10.80       25314.10        2.52
min         -52.00           0.00        0.00
25%           6.50           1.00        0.00
50%           9.50           1.72        1.16
75%          14.50           3.20        2.32
max         255.55     8005023.20      156.00

--- Estadisticas DESPUES de limpiar ---
       fare_amount  trip_distance  tip_amount
count     97292.00       97292.00    97292.00
mean         12.77          85.26        1.69
std          10.06       25663.97        2.32
min           0.01           0.01        0.00
25%           6.50           1.03        0.00
50%           9.50           1.74        1.16
75%          14.50           3.20        2.32
max       

## 4. Columnas derivadas

Creamos variables calculadas que enriquecen el analisis:

| Columna | Formula | Utilidad |
|---------|---------|----------|
| `duration_min` | (dropoff - pickup) en minutos | Duracion del viaje |
| `speed_mph` | distance / (duration / 60) | Velocidad promedio |
| `tip_pct` | tip / fare * 100 | Generosidad del pasajero |
| `pickup_hour` | hora del pickup | Patrones horarios |
| `pickup_day_of_week` | dia de la semana | Patrones semanales |

In [11]:
# duration_min ya fue calculada en el paso de limpieza

# Velocidad promedio en millas por hora
# trip_distance esta en millas, duration_min en minutos
df["speed_mph"] = df["trip_distance"] / (df["duration_min"] / 60)

# Porcentaje de propina sobre la tarifa
df["tip_pct"] = (df["tip_amount"] / df["fare_amount"]) * 100

# Componentes temporales del pickup
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["pickup_datetime"].dt.day_name()

# Filtrar velocidades fisicamente imposibles (> 80 mph en NYC)
n_antes_speed = len(df)
df = df[df["speed_mph"] <= 80]
print(f"Viajes con velocidad > 80 mph eliminados: {n_antes_speed - len(df):,}")

print(f"\nColumnas del DataFrame final: {list(df.columns)}")
print(f"Shape: {df.shape}")
df[["duration_min", "speed_mph", "tip_pct", "pickup_hour", "pickup_day_of_week"]].head(10)

Viajes con velocidad > 80 mph eliminados: 7

Columnas del DataFrame final: ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'rate_code', 'payment_type', 'fare_amount', 'tip_amount', 'tolls_amount', 'total_amount', 'duration_min', 'speed_mph', 'tip_pct', 'pickup_hour', 'pickup_day_of_week']
Shape: (97285, 18)


,duration_min,speed_mph,tip_pct,pickup_hour,pickup_day_of_week
0,7.77,9.42,29.71,20,Monday
1,5.13,6.66,0.00,20,Monday
2,5.57,9.59,0.00,17,Sunday
3,12.37,6.79,0.00,17,Monday
4,17.77,11.89,21.07,12,Sunday
5,2.17,13.29,0.00,14,Wednesday
6,5.03,7.15,15.00,19,Thursday
7,12.52,23.06,21.62,20,Tuesday
8,20.08,36.09,0.00,21,Tuesday
9,5.83,11.31,16.67,22,Friday


## 5. Perfilado del dataset limpio

Ahora que los datos estan limpios y enriquecidos, generamos un perfil completo.

In [12]:
# Estadisticas descriptivas del dataset limpio
df.describe()

,passenger_count,trip_distance,pickup_location_id,dropoff_location_id,fare_amount,tip_amount,tolls_amount,total_amount,duration_min,speed_mph,tip_pct,pickup_hour
count,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00,97285.00
mean,1.69,2.98,160.34,158.30,12.77,1.69,0.30,15.87,14.01,11.96,13.26,13.54
std,1.34,3.53,66.26,70.01,10.06,2.32,1.42,12.44,10.86,6.26,14.28,6.48
min,0.00,0.01,1.00,1.00,0.01,0.00,0.00,0.31,1.00,0.12,0.00,0.00
25%,1.00,1.03,113.00,107.00,6.50,0.00,0.00,8.75,6.75,7.77,0.00,9.00
50%,1.00,1.74,161.00,161.00,9.50,1.16,0.00,11.80,11.15,10.63,15.38,14.00
75%,2.00,3.20,231.00,231.00,14.50,2.32,0.00,17.80,17.90,14.40,22.50,19.00
max,6.00,45.50,263.00,263.00,213.21,111.00,150.08,214.01,312.70,78.68,1432.75,23.00


In [13]:
# Tipos de datos finales y uso de memoria optimizado
print("=== Tipos de datos ===")
print(df.dtypes)
print(f"\n=== Uso de memoria ===")
mem = df.memory_usage(deep=True)
print(mem)
print(f"\nTotal: {mem.sum() / 1024**2:.2f} MB")

=== Tipos de datos ===
vendor_id                           object
pickup_datetime        datetime64[us, UTC]
dropoff_datetime       datetime64[us, UTC]
passenger_count                      Int64
trip_distance                      float64
pickup_location_id                   int64
dropoff_location_id                  int64
rate_code                           object
payment_type                        object
fare_amount                        float64
tip_amount                         float64
tolls_amount                       float64
total_amount                       float64
duration_min                       float64
speed_mph                          float64
tip_pct                            float64
pickup_hour                          int32
pickup_day_of_week                  object
dtype: object

=== Uso de memoria ===
Index                   778280
vendor_id              4864250
pickup_datetime         778280
dropoff_datetime        778280
passenger_count         875565
trip_dista

In [14]:
# Distribucion de categoricas en datos limpios
print("=== Viajes por dia de la semana ===")
print(df["pickup_day_of_week"].value_counts())

print(f"\n=== Viajes por hora pico (top 5) ===")
print(df["pickup_hour"].value_counts().head())

print(f"\n=== Tipo de pago ===")
print(df["payment_type"].value_counts())

print(f"\n=== Pasajeros por viaje ===")
print(df["passenger_count"].value_counts().sort_index())

print(f"\n=== Zonas TLC de pickup mas frecuentes (top 10) ===")
print(df["pickup_location_id"].value_counts().head(10))

=== Viajes por dia de la semana ===
pickup_day_of_week
Saturday     15078
Friday       14909
Thursday     14414
Wednesday    14038
Tuesday      13453
Sunday       13157
Monday       12236
Name: count, dtype: int64

=== Viajes por hora pico (top 5) ===
pickup_hour
19    5952
20    5885
21    5781
18    5755
22    5327
Name: count, dtype: int64

=== Tipo de pago ===
payment_type
1    61041
2    35949
3      217
4       78
Name: count, dtype: int64

=== Pasajeros por viaje ===
passenger_count
0       27
1    68349
2    13867
3     4163
4     1993
5     5431
6     3455
Name: count, dtype: Int64

=== Zonas TLC de pickup mas frecuentes (top 10) ===
pickup_location_id
237    3663
161    3466
79     3381
234    3245
230    3230
236    3213
170    3209
162    3198
186    3175
48     3107
Name: count, dtype: int64


## 6. DatetimeIndex: indexacion temporal

Convertimos `pickup_datetime` en el indice del DataFrame. Esto habilita:
- Seleccion por rango de fechas con `.loc["2015-01":"2015-03"]`
- Agregaciones temporales con `.resample()`
- Graficas con ejes de tiempo automaticos

In [15]:
# Establecer pickup_datetime como indice
df = df.set_index("pickup_datetime").sort_index()
print(f"Indice: {df.index.name}")
print(f"Rango: {df.index.min()} -> {df.index.max()}")
print(f"Tipo: {df.index.dtype}")
df.head(3)

Indice: pickup_datetime
Rango: 2015-01-01 00:08:34+00:00 -> 2015-12-31 23:42:32+00:00
Tipo: datetime64[us, UTC]


,vendor_id,dropoff_datetime,passenger_count,trip_distance,pickup_location_id,dropoff_location_id,rate_code,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,duration_min,speed_mph,tip_pct,pickup_hour,pickup_day_of_week
pickup_datetime,,,,,,,,,,,,,,,,,
2015-01-01 00:08:34+00:00,2,2015-01-01 00:19:22+00:00,2,2.91,263,233,1,2,10.50,0.00,0.00,11.80,10.80,16.17,0.00,0,Thursday
2015-01-01 00:19:21+00:00,2,2015-01-01 00:21:52+00:00,1,0.56,263,141,1,2,4.00,0.00,0.00,5.30,2.52,13.35,0.00,0,Thursday
2015-01-01 00:20:45+00:00,2,2015-01-01 00:27:59+00:00,1,2.04,209,249,1,1,8.50,1.80,0.00,11.60,7.23,16.92,21.18,0,Thursday


In [16]:
# Seleccion por rango de fechas con .loc
# Ejemplo: todos los viajes del 15 de enero de 2015
viajes_15_ene = df.loc["2015-01-15"]
print(f"Viajes el 15 de enero 2015: {len(viajes_15_ene):,}")

# Ejemplo: viajes de la primera semana de febrero
viajes_feb_sem1 = df.loc["2015-02-01":"2015-02-07"]
print(f"Viajes 1-7 feb 2015: {len(viajes_feb_sem1):,}")

# Ejemplo: viajes de marzo completo
viajes_marzo = df.loc["2015-03"]
print(f"Viajes marzo 2015: {len(viajes_marzo):,}")

Viajes el 15 de enero 2015: 295
Viajes 1-7 feb 2015: 2,084
Viajes marzo 2015: 8,816


## 7. Resample: agregaciones temporales

Con el DatetimeIndex podemos agregar los viajes por hora, por dia, por semana, etc. Esto revela patrones temporales en la demanda y las tarifas.

In [17]:
# Conteo de viajes por hora
viajes_por_hora = df.resample("h").size()
viajes_por_hora.name = "n_viajes"

print("Viajes por hora (primeras 24 horas):")
print(viajes_por_hora.head(24))
print(f"\nPromedio de viajes por hora: {viajes_por_hora.mean():.1f}")

Viajes por hora (primeras 24 horas):
pickup_datetime
2015-01-01 00:00:00+00:00    18
2015-01-01 01:00:00+00:00    19
2015-01-01 02:00:00+00:00    10
2015-01-01 03:00:00+00:00    13
2015-01-01 04:00:00+00:00    10
2015-01-01 05:00:00+00:00     8
2015-01-01 06:00:00+00:00     9
2015-01-01 07:00:00+00:00     2
2015-01-01 08:00:00+00:00     2
2015-01-01 09:00:00+00:00    10
2015-01-01 10:00:00+00:00     7
2015-01-01 11:00:00+00:00    11
2015-01-01 12:00:00+00:00    12
2015-01-01 13:00:00+00:00    13
2015-01-01 14:00:00+00:00    16
2015-01-01 15:00:00+00:00    15
2015-01-01 16:00:00+00:00    18
2015-01-01 17:00:00+00:00    12
2015-01-01 18:00:00+00:00    15
2015-01-01 19:00:00+00:00    11
2015-01-01 20:00:00+00:00     9
2015-01-01 21:00:00+00:00    11
2015-01-01 22:00:00+00:00    10
2015-01-01 23:00:00+00:00     9
Freq: h, Name: n_viajes, dtype: int64

Promedio de viajes por hora: 11.1


In [18]:
# Conteo de viajes por dia
viajes_por_dia = df.resample("D").size()
viajes_por_dia.name = "n_viajes"

print("Viajes por dia (primeros 10 dias):")
print(viajes_por_dia.head(10))
print(f"\nPromedio diario: {viajes_por_dia.mean():.0f} viajes/dia")

Viajes por dia (primeros 10 dias):
pickup_datetime
2015-01-01 00:00:00+00:00    270
2015-01-02 00:00:00+00:00    208
2015-01-03 00:00:00+00:00    278
2015-01-04 00:00:00+00:00    226
2015-01-05 00:00:00+00:00    239
2015-01-06 00:00:00+00:00    227
2015-01-07 00:00:00+00:00    294
2015-01-08 00:00:00+00:00    299
2015-01-09 00:00:00+00:00    296
2015-01-10 00:00:00+00:00    321
Freq: D, Name: n_viajes, dtype: int64

Promedio diario: 267 viajes/dia


In [19]:
# Tarifa promedio diaria
tarifa_diaria = df.resample("D")["fare_amount"].mean()

print("Tarifa promedio diaria (primeros 10 dias):")
print(tarifa_diaria.head(10))
print(f"\nTarifa promedio global: ${tarifa_diaria.mean():.2f}")

Tarifa promedio diaria (primeros 10 dias):
pickup_datetime
2015-01-01 00:00:00+00:00   14.50
2015-01-02 00:00:00+00:00   11.80
2015-01-03 00:00:00+00:00   12.04
2015-01-04 00:00:00+00:00   12.42
2015-01-05 00:00:00+00:00   12.82
2015-01-06 00:00:00+00:00   11.26
2015-01-07 00:00:00+00:00   10.95
2015-01-08 00:00:00+00:00   11.76
2015-01-09 00:00:00+00:00   11.34
2015-01-10 00:00:00+00:00   11.69
Freq: D, Name: fare_amount, dtype: float64

Tarifa promedio global: $12.78


In [20]:
# Resumen mensual combinado
resumen_mensual = df.resample("ME").agg(
    n_viajes=("fare_amount", "size"),
    fare_mean=("fare_amount", "mean"),
    fare_median=("fare_amount", "median"),
    distance_mean=("trip_distance", "mean"),
    duration_mean=("duration_min", "mean"),
    tip_pct_mean=("tip_pct", "mean"),
)

print("Resumen mensual:")
resumen_mensual

Resumen mensual:


,n_viajes,fare_mean,fare_median,distance_mean,duration_mean,tip_pct_mean
pickup_datetime,,,,,,
2015-01-31 00:00:00+00:00,8378,11.97,9.00,2.85,12.45,12.98
2015-02-28 00:00:00+00:00,8240,12.17,9.50,2.80,13.45,13.60
2015-03-31 00:00:00+00:00,8816,12.46,9.50,2.89,13.85,13.30
2015-04-30 00:00:00+00:00,8676,12.63,9.50,2.92,13.88,13.21
2015-05-31 00:00:00+00:00,8932,13.05,9.50,3.05,14.44,13.33
2015-06-30 00:00:00+00:00,8278,13.13,10.00,3.06,14.55,13.18
2015-07-31 00:00:00+00:00,7656,12.73,9.50,2.98,13.81,13.08
2015-08-31 00:00:00+00:00,7497,12.75,9.50,3.02,13.67,12.82
2015-09-30 00:00:00+00:00,7457,13.39,10.00,3.17,14.59,13.34


## Resumen y conclusiones

### Lo que aprendimos:
1. **Carga desde BigQuery**: usar `run_query()` con cache evita consultas repetidas y controla costos
2. **Limpieza de datos reales**: zonas TLC invalidas, tarifas negativas y duraciones imposibles son comunes (~5-15% de los datos)
3. **Columnas derivadas**: `duration_min`, `speed_mph` y `tip_pct` transforman datos crudos en metricas de negocio
4. **Perfilado**: `describe()`, `dtypes`, `memory_usage()` y `value_counts()` son las herramientas esenciales
5. **DatetimeIndex + resample**: permiten analizar patrones temporales (por hora, dia, mes)

### Respuesta a la pregunta de negocio:
> **¿Cual es el perfil tipico de un viaje en taxi en NYC?**
>
> Con los datos limpios podemos ver las medianas y medias de cada variable clave.
> El siguiente paso es visualizar estas distribuciones y buscar patrones
> por hora del dia, dia de la semana y zona geografica (usando los IDs de zonas TLC).

### Siguiente notebook:
En el proximo notebook exploraremos visualmente estas distribuciones y patrones temporales.